In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import f1_score # f1 score
from sklearn.metrics import fbeta_score # f2 score
from sklearn.metrics import recall_score # recall score

In [2]:
use_cuda = True
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")
print(f"Using device: {device}")

CUDA available: True
Using device: cuda


In [3]:
df = pd.read_csv('./ieee-fraud-detection/train_transaction.csv')
display(df)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def preprocess(df):
    df = df.copy()
    if 'TransactionDT' in df.columns:
        # hour of the day
        df['TransactionHour'] = (df['TransactionDT'] // 3600) % 24
        # day since first transaction
        df['TransactionDay'] = df['TransactionDT'] // (3600*24)
        # day of the week
        df['TransactionWeekday'] = df['TransactionDay'] % 7
        # fill missing just in case
        df[['TransactionHour','TransactionDay','TransactionWeekday']] = df[['TransactionHour','TransactionDay','TransactionWeekday']].fillna(-999)
        
    features_to_drop = [
        'TransactionID',
        'TransactionDT',
    ]
    df.drop(columns=[c for c in features_to_drop if c in df.columns], inplace=True)

    # Numeric columns
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if 'isFraud' in numeric_cols:
        numeric_cols.remove('isFraud')
    df[numeric_cols] = df[numeric_cols].fillna(-999)
    
    # Categorical columns
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    df[categorical_cols] = df[categorical_cols].fillna('Unknown')
    
    for col in categorical_cols:
        df[col] = df[col].astype('category').cat.codes
    
    y = df['isFraud']
    X = df.drop(columns=['isFraud'])
    
    return X, y

In [5]:
X, y = preprocess(df)
print(f"Features shape: {X.shape}, Target shape: {y.shape}")

Features shape: (590540, 394), Target shape: (590540,)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(
    n_estimators=5000,
    max_depth=8,
    learning_rate=0.025,
    scale_pos_weight=27.4,
    n_jobs=-1,
    eval_metric='auc'
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

y_pred = model.predict_proba(X_val)[:,1]
roc_auc = roc_auc_score(y_val, y_pred)
pr_auc = average_precision_score(y_val, y_pred)
print(f"Validation ROC-AUC: {roc_auc:.4f}")
print(f"Validation PR-AUC: {pr_auc:.4f}")

[0]	validation_0-auc:0.85897
[50]	validation_0-auc:0.89545
[100]	validation_0-auc:0.91162
[150]	validation_0-auc:0.91981
[200]	validation_0-auc:0.92558
[250]	validation_0-auc:0.93176
[300]	validation_0-auc:0.93576
[350]	validation_0-auc:0.93939
[400]	validation_0-auc:0.94242
[450]	validation_0-auc:0.94446
[500]	validation_0-auc:0.94649
[550]	validation_0-auc:0.94881
[600]	validation_0-auc:0.95018
[650]	validation_0-auc:0.95154
[700]	validation_0-auc:0.95279
[750]	validation_0-auc:0.95390
[800]	validation_0-auc:0.95464
[850]	validation_0-auc:0.95552
[900]	validation_0-auc:0.95641
[950]	validation_0-auc:0.95683
[1000]	validation_0-auc:0.95743
[1050]	validation_0-auc:0.95780
[1100]	validation_0-auc:0.95820
[1150]	validation_0-auc:0.95867
[1200]	validation_0-auc:0.95918
[1250]	validation_0-auc:0.95941
[1300]	validation_0-auc:0.95978
[1350]	validation_0-auc:0.96031
[1400]	validation_0-auc:0.96060
[1450]	validation_0-auc:0.96087
[1500]	validation_0-auc:0.96122
[1550]	validation_0-auc:0.96160

In [7]:
# Find f1, f2 and recall score using best threshold
y_prob = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.01, 0.9, 100)
f1_scores = []
f2_scores = []
for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    f1_scores.append(f1_score(y_val, y_pred))
    f2_scores.append(fbeta_score(y_val, y_pred, beta=2))

# Best thresholds
best_f1_thresh = thresholds[np.argmax(f1_scores)]
best_f2_thresh = thresholds[np.argmax(f2_scores)]

best_f1 = max(f1_scores)
best_f2 = max(f2_scores)

y_pred_best = (y_prob > best_f2_thresh).astype(int)

recall = recall_score(y_val, y_pred_best)

print(f"Best F1 threshold: {best_f1_thresh:.2f}, F1: {best_f1:.4f}")
print(f"Best F2 threshold: {best_f2_thresh:.2f}, F2: {best_f2:.4f}")
print(f"Recall (at best F2 threshold): {recall:.4f}")

Best F1 threshold: 0.73, F1: 0.7777
Best F2 threshold: 0.51, F2: 0.7621
Recall (at best F2 threshold): 0.7861


In [8]:
# Find f1, f2 and recall score using threshold of 0.5
y_pred = (model.predict_proba(X_val)[:,1] > 0.5).astype(int)
recall = recall_score(y_val, y_pred)
print("F1-score:", f1_score(y_val, y_pred))
print("F2-score:", fbeta_score(y_val, y_pred, beta=2))
print("Recall:", recall)

F1-score: 0.7235835458476549
F2-score: 0.7617424596134093
Recall: 0.7894991531575127
